# Módulos estándar · `datetime`

Fechas, horas, formatos y diferencias de tiempo.

## 1. Partes de la fecha actual

Obtener el día, mes, año, hora, minuto y timestamp actuales.

In [1]:
from datetime import datetime

ahora = datetime.now()
print(ahora.day, ahora.month, ahora.year, ahora.hour, ahora.minute, ahora.timestamp())

6 11 2026 22 2 1794002533.511745


## 2. Formatear y analizar fechas

Mostrar la fecha actual con el formato `"%m/%d/%Y, %H:%M:%S"` y convertir la cadena `"5 December, 2019"` en un objeto `datetime`.

In [2]:
print(ahora.strftime("%m/%d/%Y, %H:%M:%S"))
print(datetime.strptime("5 December, 2019", "%d %B, %Y"))

11/06/2026, 22:02:13
2019-12-05 00:00:00


## 3. Diferencias de tiempo

Calcular cuánto falta para el próximo Año Nuevo y cuánto tiempo ha pasado desde el 1 de enero de 1970.

In [3]:
año_nuevo = datetime(ahora.year + 1, 1, 1)
print("Hasta Año Nuevo:", año_nuevo - ahora)
print("Desde 1970:", ahora - datetime(1970, 1, 1))

Hasta Año Nuevo: 55 days, 1:57:46.488255
Desde 1970: 20763 days, 22:02:13.511745


## 4. `date`, `datetime` y `timedelta`

A partir del 6 de noviembre de 2026 a las 9:30, calcular la fecha 100 días después, la de hace 6 semanas y cuántas horas y minutos faltan para las 14:15 del mismo día.

In [4]:
from datetime import date, time, timedelta

inicio = datetime(2026, 11, 6, 9, 30)
print("+100 días:", inicio + timedelta(days=100))
print("-6 semanas:", (inicio - timedelta(weeks=6)).date())

salida = datetime.combine(inicio.date(), time(14, 15))
horas, resto = divmod(int((salida - inicio).total_seconds()), 3600)
print(f"Faltan {horas} h {resto // 60} min")
assert (horas, resto // 60) == (4, 45)
assert (inicio + timedelta(days=100)).date() == date(2027, 2, 14)

+100 días: 2027-02-14 09:30:00
-6 semanas: 2026-09-25
Faltan 4 h 45 min


## 5. `strftime` y `strptime`

Convertir `"06/11/2026 14:30"` en `datetime`, mostrarlo como `"viernes, 6 de noviembre de 2026, 14:30"` sin depender del *locale* y comprobar que el formato numérico es reversible.

In [5]:
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio",
         "agosto", "septiembre", "octubre", "noviembre", "diciembre"]

def en_español(f):
    return f"{DIAS[f.weekday()]}, {f.day} de {MESES[f.month - 1]} de {f.year}, {f:%H:%M}"

FORMATO = "%d/%m/%Y %H:%M"
cita = datetime.strptime("06/11/2026 14:30", FORMATO)
print(repr(cita))
print(en_español(cita))
print(cita.strftime("%Y%m%d-%H%M"), "· día", cita.strftime("%j"), "del año")
assert cita.strftime(FORMATO) == "06/11/2026 14:30"
assert en_español(cita) == "viernes, 6 de noviembre de 2026, 14:30"

datetime.datetime(2026, 11, 6, 14, 30)
viernes, 6 de noviembre de 2026, 14:30
20261106-1430 · día 310 del año


## 6. Edad exacta

Escribir `edad(nacimiento, hoy)`, que devuelva los años, meses y días cumplidos, y comprobarla con 10 000 pares de fechas aleatorias reconstruyendo `hoy` a partir del resultado.

In [6]:
import calendar
import random

def sumar_meses(f, n):
    año, mes = divmod(f.year * 12 + f.month - 1 + n, 12)
    return date(año, mes + 1, min(f.day, calendar.monthrange(año, mes + 1)[1]))

def edad(nacimiento, hoy):
    meses = (hoy.year - nacimiento.year) * 12 + hoy.month - nacimiento.month
    if sumar_meses(nacimiento, meses) > hoy:
        meses -= 1
    return meses // 12, meses % 12, (hoy - sumar_meses(nacimiento, meses)).days

print(edad(date(1990, 5, 23), date(2026, 11, 6)))

random.seed(0)
for _ in range(10_000):
    nacimiento = date(1950, 1, 1) + timedelta(days=random.randrange(25_000))
    hoy = nacimiento + timedelta(days=random.randrange(40_000))
    años, meses, dias = edad(nacimiento, hoy)
    assert sumar_meses(nacimiento, años * 12 + meses) + timedelta(days=dias) == hoy
    assert sumar_meses(nacimiento, años * 12 + meses + 1) > hoy
    if nacimiento.day <= 28:  # el 29/2 cumple el 28/2 en años no bisiestos
        assert años == hoy.year - nacimiento.year - ((hoy.month, hoy.day) < (nacimiento.month, nacimiento.day))

(36, 5, 14)


## 7. Días laborables

Contar los días laborables (de lunes a viernes y sin festivos) entre dos fechas, con el inicio incluido y el fin excluido, tomando como festivos los nacionales de fecha fija y el Día de Canarias (sin traslados), y comprobarlo con `numpy.busday_count`.

In [7]:
import numpy as np

FESTIVOS = {date(2026, 1, 1), date(2026, 1, 6), date(2026, 5, 1), date(2026, 5, 30), date(2026, 8, 15),
            date(2026, 10, 12), date(2026, 11, 1), date(2026, 12, 6), date(2026, 12, 8), date(2026, 12, 25)}

def dias_laborables(inicio, fin, festivos=FESTIVOS):
    dias = (inicio + timedelta(days=i) for i in range((fin - inicio).days))
    return sum(d.weekday() < 5 and d not in festivos for d in dias)

n = dias_laborables(date(2026, 1, 1), date(2027, 1, 1))
print("Laborables en 2026:", n, "· en noviembre:", dias_laborables(date(2026, 11, 1), date(2026, 12, 1)))
assert n == np.busday_count("2026-01-01", "2027-01-01", holidays=[d.isoformat() for d in FESTIVOS])

Laborables en 2026: 255 · en noviembre: 21


## 8. Calendario de un mes

Mostrar el calendario de noviembre de 2026 con `calendar.TextCalendar` y escribir `calendario_es(año, mes)`, que lo dibuje en español a partir de `calendar.monthcalendar`.

In [8]:
print(calendar.TextCalendar(calendar.MONDAY).formatmonth(2026, 11))

def calendario_es(año, mes):
    lineas = [f"{MESES[mes - 1].capitalize()} {año}".center(20), "Lu Ma Mi Ju Vi Sá Do"]
    for semana in calendar.monthcalendar(año, mes):
        lineas.append(" ".join(f"{d:2}" if d else "  " for d in semana))
    return "\n".join(lineas)

print(calendario_es(2026, 11))
dias = [d for semana in calendar.monthcalendar(2026, 11) for d in semana if d]
assert dias == list(range(1, calendar.monthrange(2026, 11)[1] + 1))

   November 2026
Mo Tu We Th Fr Sa Su
                   1
 2  3  4  5  6  7  8
 9 10 11 12 13 14 15
16 17 18 19 20 21 22
23 24 25 26 27 28 29
30

   Noviembre 2026   
Lu Ma Mi Ju Vi Sá Do
                   1
 2  3  4  5  6  7  8
 9 10 11 12 13 14 15
16 17 18 19 20 21 22
23 24 25 26 27 28 29
30                  


## 9. Zonas horarias

Con `zoneinfo`, pasar una reunión a las 10:00 de Madrid a la hora de Canarias en invierno y en verano, comprobar que la diferencia es siempre de una hora y encontrar los días de 2026 en los que cambia la hora (el último domingo de marzo y de octubre).

In [9]:
from zoneinfo import ZoneInfo

MADRID = ZoneInfo("Europe/Madrid")
CANARIAS = ZoneInfo("Atlantic/Canary")

for reunion in (datetime(2026, 1, 15, 10, tzinfo=MADRID), datetime(2026, 7, 15, 10, tzinfo=MADRID)):
    en_canarias = reunion.astimezone(CANARIAS)
    print(f"{reunion:%d/%m %H:%M %Z} -> {en_canarias:%H:%M %Z} (UTC{en_canarias:%z})")
    assert reunion == en_canarias
    assert reunion.utcoffset() - en_canarias.utcoffset() == timedelta(hours=1)

def horario_verano(d):
    return bool(datetime.combine(d, time(12), CANARIAS).dst())

dias_2026 = [date(2026, 1, 1) + timedelta(days=i) for i in range(365)]
cambios = [d for d in dias_2026[1:] if horario_verano(d) != horario_verano(d - timedelta(days=1))]
print("Cambios de hora en 2026:", [d.isoformat() for d in cambios])
ultimo_domingo = lambda mes: max(d for d in dias_2026 if d.month == mes and d.weekday() == 6)
assert cambios == [ultimo_domingo(3), ultimo_domingo(10)]

15/01 10:00 CET -> 09:00 WET (UTC+0000)


15/07 10:00 CEST -> 09:00 WEST (UTC+0100)
Cambios de hora en 2026: ['2026-03-29', '2026-10-25']


## 10. ISO 8601

Pasar fechas con zona horaria a texto ISO 8601 y de vuelta con `isoformat` y `fromisoformat`, y obtener la semana ISO de una fecha con `isocalendar`.

In [10]:
momento = datetime(2026, 11, 6, 14, 30, tzinfo=CANARIAS)
texto = momento.isoformat()
print(texto)
assert datetime.fromisoformat(texto) == momento
assert datetime.fromisoformat("2026-11-06T15:30:00+01:00") == momento
assert datetime.fromisoformat("2026-11-06T14:30Z") == momento

año, semana, dia = date(2026, 11, 6).isocalendar()
print(f"Semana ISO {semana} de {año}, día {dia}")
assert date.fromisoformat(f"{año}-W{semana:02}-{dia}") == date(2026, 11, 6)
print("El 1/1/2027 pertenece a la semana", date(2027, 1, 1).isocalendar()[:2])

2026-11-06T14:30:00+00:00
Semana ISO 45 de 2026, día 5
El 1/1/2027 pertenece a la semana (2026, 53)


## 11. Bisiestos y día de la semana

Escribir `es_bisiesto(año)` y `dia_semana(fecha)` (congruencia de Zeller), y comprobarlas con `calendar.isleap` y `date.weekday` para todos los días de 1900 a 2100.

In [11]:
def es_bisiesto(año):
    return año % 4 == 0 and (año % 100 != 0 or año % 400 == 0)

def dia_semana(f):
    d, m, a = f.day, f.month, f.year
    if m < 3:
        m, a = m + 12, a - 1
    h = (d + 13 * (m + 1) // 5 + a + a // 4 - a // 100 + a // 400) % 7
    return (h + 5) % 7  # Zeller da 0 = sábado; weekday usa 0 = lunes

assert all(es_bisiesto(a) == calendar.isleap(a) for a in range(1, 10_000))
f = date(1900, 1, 1)
while f <= date(2100, 12, 31):
    assert dia_semana(f) == f.weekday()
    f += timedelta(days=1)

print("Bisiestos entre 1896 y 1912:", [a for a in range(1896, 1913) if es_bisiesto(a)])
print("Bisiestos en el siglo XXI:", calendar.leapdays(2001, 2101))
print("El 6/11/2026 es", DIAS[dia_semana(date(2026, 11, 6))])

Bisiestos entre 1896 y 1912: [1896, 1904, 1908, 1912]
Bisiestos en el siglo XXI: 24
El 6/11/2026 es viernes
